# AlphaStableFlowLinear minimal run

In [8]:
import torch
from cauda.datasets import fetch_synthetic_data
from cauda.flow import AlphaStableFlowLinear
from cauda.metrics import msle_at_quantile
from cauda.model import LightNet
from cauda.training import train

In [9]:
dim = 2
n_samples = 1000
alpha = 1.5
n_steps = 20
batch_size = 128
n_epochs = 5
lr = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

In [10]:
x_train, _, x_ref = fetch_synthetic_data("alpha_stable",
                                         n_samples=n_samples,
                                         dim=dim,
                                         alpha=alpha,
                                         device=device,
                                         dtype=dtype,
                                         )

In [11]:
net = LightNet(dim=dim).to(device=device, dtype=dtype)
generator = AlphaStableFlowLinear(net=net,
                                  dim=dim,
                                  alpha=alpha,
                                  n_steps=n_steps,
                                  device=device,
                                  dtype=dtype,
                                  )

In [14]:

_, _ = train(generator,
      target_data=x_train.clone(),
      batch_size=batch_size,
      n_epochs=n_epochs,
      lr=lr,
      use_adamw=False,
      grad_clip_norm=None,
      num_workers=0,
      device=device,
      )

In [16]:
x = generator.sample(n_samples=n_samples)
x_ref = x_ref.clone()

print(f"MSLE-95 = {msle_at_quantile(x.abs(), x_ref.abs()):.4f}")

MSLE-95 = 0.0058
